# VFPred
### A Fusion of Signal Processing and Machine Learning techniques in Detecting Ventricular Fibrillation from ECG Signals

**Authors:** Nabil Ibtehaz, M. Saifur Rahman, M. Sohel Rahman  
**Published in:** *Biomedical Signal Processing and Control*, vol. 49, pp. 349–359, 2019, Elsevier.

---

VFPred is a robust algorithm for detecting **Ventricular Fibrillation (VF)** from ECG signals. VF is one of the most dangerous cardiac arrhythmias, responsible for sudden cardiac arrests: it occurs when the heart quivers instead of pumping due to disturbances in the electrical activity of the ventricles. Fast and accurate detection is life-critical.

The algorithm bridges two traditionally separate worlds: **signal processing** (EMD + DFT) and **machine learning** (SVM). Prior signal-processing methods set a single threshold on a computed parameter and suffer from class imbalance or selection bias. Prior ML approaches treat ECG parameters as black-box features. VFPred instead starts from a medical observation — *QRS complexes are absent in VF signals* — formalises it via EMD/DFT feature engineering, and then trains an SVM to handle the real-world diversity of the data.

**Key results (10-fold cross-validation on VFDB + CUDB, T_e = 5 s):**
- Sensitivity: **99.99% ± 0.016%**
- Specificity: **98.40% ± 0.19%**
- G-Mean Accuracy: **99.19% ± 0.09%**

The notebook walks through the complete pipeline end-to-end.

## Importing Libraries

Importing the required modules

### Overview
This cell imports every external dependency used throughout the pipeline. The libraries span four domains: numerical computing, signal processing, machine learning, and data handling.

### Step-by-step
1. **Standard library & I/O** — `os`, `os.path`, `pickle`: file-system navigation and serialising Python objects to disk (`.p` pickle files are used extensively for intermediate results).
2. **PhysioNet access** — `wfdb`: the WFDB (WaveForm DataBase) package fetches ECG records and annotations directly from PhysioNet servers (Goldberger et al., 2000).
3. **Numerical computing** — `numpy`, `pandas`: array maths and tabular data manipulation.
4. **Signal processing** — `scipy.fftpack.fft/ifft/fftshift`: Fast Fourier Transform used to move into the frequency domain; `PyEMD.EMD`: the Empirical Mode Decomposition library (Laszuk, 2017) that decomposes ECG episodes into Intrinsic Mode Functions.
5. **Visualisation** — `matplotlib`, `seaborn`: plotting spectra, histograms, and feature importance charts.
6. **Progress tracking** — `tqdm`: progress bars for the long-running batch processing loops.
7. **Machine learning** — `sklearn.svm.SVC`: the RBF-kernel Support Vector Machine classifier; `sklearn.preprocessing`, `sklearn.decomposition`: feature normalisation utilities; `sklearn.ensemble.ExtraTreesClassifier`, `RandomForestClassifier`: ensemble methods used for feature ranking; `sklearn.model_selection.KFold`, `StratifiedKFold`: cross-validation splitters.
8. **Class imbalance** — `imblearn.over_sampling.SMOTE`: Synthetic Minority Over-Sampling Technique (Chawla et al., 2002) for generating synthetic VF samples to balance the training set.
9. **Project-local modules** — `data_structures` (EcgSignal, Annotation, Features data classes), `helper_function.cosineSimilarity` (cosine similarity helper).

In [1]:
# conda install -c conda-forge emd-signal seaborn imbalanced-learn

import os
import os.path
import pickle
import wfdb

import numpy as np
import pandas as pd

from scipy.fftpack import ifft
from scipy.fftpack import fft , fftshift
from PyEMD import EMD

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import seaborn
from tqdm import tqdm

from sklearn import svm
from sklearn import preprocessing
from sklearn import decomposition
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE

from data_structures import Features, EcgSignal,Annotation
from helper_function import cosineSimilarity

## Data Structures

The three classes defined in [`data_structures.py`](data_structures.py) are the shared currency passed between every stage of the pipeline. Each stage reads objects produced by the previous stage and writes enriched versions for the next.

---

### `Annotation`
Holds the expert beat-level labels for a recording, as provided by the PhysioNet databases.

| Field | Type | Description |
|---|---|---|
| `index` | `array[int]` | Sample indices where annotation markers occur (sparse — one per beat or rhythm change) |
| `label` | `array[str]` | Corresponding label at each index (e.g. `'N'` = normal, `'['` = VF onset, `']'` = VF end, `'V'` = PVC) |

The sparse `(index, label)` pairs are later expanded into a dense per-sample array by `createAnnotationArray` and `createDistributedAnnotations` in `data_handling.py`.

---

### `EcgSignal`
Produced by **Processing the dataset** (`processData`). Wraps one fixed-length episode window extracted from a raw recording.

| Field | Type | Description |
|---|---|---|
| `signal` | `ndarray[float]`, shape `(T_e × Fs,)` | Raw ADC samples for the episode (e.g. 1250 samples for T_e = 5 s at 250 Hz) |
| `annotation` | `Annotation` | Sparse beat annotations covering this episode's time range |
| `Fs` | `int` | Sampling frequency in Hz (250 Hz for both VFDB and CUDB) |
| `channel` | `int` | Channel number (always 1 for VFDB; CUDB is single-channel) |
| `source` | `str` | Database origin (`'MITMVA'` or `'cudb'`) |

Serialised to `Pickles/MITMVAdb/<file>E<episode>C<channel>.p` and `Pickles/cudb/…`.

---

### `Features`
Produced by **Signal Processing** (`batchSignalProcessing`) and enriched by **Annotating** (`labelEpisodes`). Contains all frequency-domain representations and metadata needed by the machine learning stage.

| Field | Type | Description |
|---|---|---|
| `Signal_FFT` | `ndarray[complex]`, shape `(N,)` | DFT of the filtered ECG episode (centred via `fftshift`), N = T_e × Fs |
| `IMF1_FFT` | `ndarray[complex]`, shape `(N,)` | DFT of the selected IMF component (IMF₁, or IMF₁+IMF₂ if NLCR ≤ β) |
| `R_FFT` | `ndarray[complex]`, shape `(N,)` | DFT of the Residue component R = signal − IMF |
| `Fs` | `int` | Sampling frequency (preserved for downstream bin-to-frequency conversion) |
| `imf1_Sim` | `float` | Time-domain cosine similarity of signal and IMF₁ — the scalar feature explored by Anas et al. (2011) before the authors moved to frequency-domain features |
| `R_Sim` | `float` | Time-domain cosine similarity of signal and Residue |
| `imf12` | `float` | Cosine similarity between IMF₁ and IMF₂ (retained but marked unnecessary in the paper) |
| `imf23` | `float` | Cosine similarity between IMF₂ and IMF₃ (retained but marked unnecessary) |
| `label` | `list[int]`, length 10 | Multi-threshold VF label: `label[k] = 1` if ≥ (k+1)×10% of the episode samples are VF, for k = 0…9. Index 9 (`percent=100`) is used in training |
| `file` | `int` | Source record identifier (e.g. 418 for VFDB, 0–34 for CUDB) |
| `episode` | `int` | Episode index within the record (window number) |
| `channel` | `int` | Channel number |

Serialised to `Pickles/MITMVAdbFFT/F<file>E<episode>C<channel>.p` and `Pickles/cudbFFT/…`.

> **Pipeline flow:**  
> `Annotation` → packed into `EcgSignal` (raw window) → `Features.Signal/IMF1/R_FFT` (frequency domain) → `Features.label` (annotation) → `createData` assembles the 2N-dimensional ML feature vector from `Signal_FFT`, `IMF1_FFT`, and `R_FFT`.


## Downloading the dataset

Downloading the dataset from physionet

### Overview
This step retrieves the two benchmark ECG databases used throughout the paper (§ Dataset) directly from PhysioNet (Goldberger et al., 2000). Both databases contain 30-second to 8-minute multi-channel ECG recordings of subjects who experienced episodes of Ventricular Tachycardia (VT), Ventricular Flutter, and Ventricular Fibrillation (VF), along with expert beat-level annotations.

### Databases
| Database | Records | Duration | Fs | Description |
|---|---|---|---|---|
| **MIT-BIH VFDB** (MITMVA) | 22 recordings (IDs 418–430, 602–615) | ~30 min each | 250 Hz | Malignant Ventricular Arrhythmia Database (Greenwald, 1986) |
| **CUDB** | 35 recordings (cu01–cu35) | ~8 min each | 250 Hz | Creighton University Ventricular Tachyarrhythmia Database (Nolle et al., 1986), pre-filtered with a 2nd-order Bessel LPF at 70 Hz, quantised at 12-bit over 10 V range |

### Step-by-step (`data_handling.downloadData`)
1. For each record ID in VFDB (418–430, 602–615) and CUDB (cu01–cu35), `wfdb.get_record_list` enumerates the files belonging to that record on PhysioNet.
2. `wfdb.dl_files` downloads the raw signal (`.dat`), header (`.hea`), and annotation (`.atr`) files into local `database/mitMVAdb/` and `database/cudb/` subdirectories.
3. A progress message is printed for each record and a final "Finished downloading" confirmation is shown.

> The datasets are publicly available at [PhysioNet](https://physionet.org) under open access.

In [2]:
from data_handling import downloadData

downloadData()

Generating record list for: 418
Generating record list for: 419
Generating record list for: 420
Generating record list for: 421
Generating record list for: 422
Generating record list for: 423
Generating record list for: 424
Generating record list for: 425
Generating record list for: 426
Generating record list for: 427
Generating record list for: 428
Generating record list for: 429
Generating record list for: 430
Generating record list for: 602
Generating record list for: 605
Generating record list for: 607
Generating record list for: 609
Generating record list for: 610
Generating record list for: 611
Generating record list for: 612
Generating record list for: 614
Generating record list for: 615
Generating list of all files for: 418
Generating list of all files for: 419
Generating list of all files for: 420
Generating list of all files for: 421
Generating list of all files for: 422
Generating list of all files for: 423
Generating list of all files for: 424
Generating list of all files f

## Processing the dataset

Breaking the long ECG signals into episodes and saving them

### Overview
Raw ECG recordings are long (up to 30 min) continuous signals. VFPred operates on short, fixed-length *episodes*. This step segments each recording into overlapping windows of length **T_e = 5 seconds**, shifted by 1 second, and serialises each window as a separate `EcgSignal` object. The episode length T_e was chosen as a balance between detection speed and accuracy after comparing T_e = 2 s, 5 s, and 8 s (§ SVM Parameter Tuning): 5 s yields a G-Mean Accuracy of 91.97 % on the validation set, whereas 8 s gives 95.10 % at the cost of slower real-time response.

### Step-by-step (`data_handling.processData` → `processMITMVADB` + `processCUDB`)

**MITMVA DB processing (`processMITMVADBFile`):**
1. Read the raw signal and annotation using `wfdb.rdrecord` / `wfdb.rdann`.
2. Only **channel 1** data from VFDB is used to avoid redundancy from the two-channel recordings.
3. Slide a window of `T_e × Fs = 5 × 250 = 1250` samples across the recording, stepping 250 samples (1 s) at a time.
4. Annotation symbols are parsed; special symbols (rhythm changes, noise markers) are handled to create a point-wise `Annotation` object containing per-sample beat labels.
5. Episodes marked entirely as noise are discarded.
6. Each episode is saved as a pickle file: `Pickles/MITMVAdb/<file>E<episode>C<channel>.p` containing an `EcgSignal(signal, annotation, Fs, channel, source)` object.
7. **Result:** 282 episodes extracted from MITMVA DB in ~39 min; 40 episodes from CUDB in ~10 min.

**CUDB processing (`processCUDBFile`):**
1. Follows the same sliding-window strategy.
2. CUDB annotations use a different convention: `N` marks Normal Sinus Rhythm; `[` and `]` delimit VF onset and termination. A dedicated annotation parser (`createCUDBAnnotation`) translates these into the unified VF/NotVF/NSR label scheme.
3. Episodes saved as `Pickles/cudb/<file>E<episode>C1.p`.

> This segmentation strategy mirrors the practice established in the literature (§ Dataset) and ensures each episode has a consistent sample length for subsequent FFT computation.

In [3]:
from data_handling import processData

processData(Te=5)

Processing MITMVAdb files


100%|██████████| 300/300 [00:53<00:00,  5.65it/s]


Processing CUdb files


100%|██████████| 40/40 [00:31<00:00,  1.28it/s]


## Signal Processing

Performing the signal processing pipeline on all the ECG episodes

### Overview
This is the core feature-extraction stage. For each serialised ECG episode, the algorithm applies a **four-step filtering pipeline** followed by **Empirical Mode Decomposition (EMD)** and **Discrete Fourier Transform (DFT)**. The extracted frequency-domain similarity vectors become the input features for the machine learning classifier. This corresponds to §§ Signal Preprocessing and Filtering, Analyzing the Oscillatory Characteristics, and Extracting Frequency Information from Oscillations of the paper.

The key insight driving the feature design is that **QRS complexes are absent in VF signals** (Jones & Owens, 2009). In normal rhythms, sharp QRS peaks introduce high-frequency oscillations captured by EMD's first IMF. In VF, the signal lacks these peaks, so IMF₁ closely tracks the original signal instead of diverging from it. This difference is quantified in the frequency domain.

### Step-by-step (`signal_processing.batchSignalProcessing` → `processSignal`)

**A. Signal Filtering (`filtering`)**  
Applied to each 1250-sample (5 s × 250 Hz) window:
1. **Mean subtraction** — Remove the DC component (zero-mean the signal).
2. **Moving-average filter (order 5)** — Suppress muscle noise and high-frequency interspersions.
3. **High-pass Butterworth filter (f_c = 1 Hz, order 1)** — Eliminate baseline wander / drift.
4. **Low-pass Butterworth filter (f_c = 20 Hz, order 12)** — Remove unnecessary high-frequency content above the physiologically relevant band.  
This pipeline follows Amann et al. (2005) as modified by Anas et al. (2011).

**B. EMD Decomposition**  
`PyEMD.EMD().emd(filtered_signal)` decomposes the filtered signal x(n) into:  
x(n) = IMF₁(n) + IMF₂(n) + R(n)  
where IMF components are listed in decreasing order of oscillatory frequency (Huang et al., 1998).

**C. Noise Level Crossing Ratio (NLCR)**  
To determine if IMF₁ contains signal content or noise (scheme from Anas et al., 2011):
1. Compute noise level: V_n = α × max(x(n)), α = 0.05.
2. Identify low-amplitude samples: n_L = {t : |IMF₁(t)| ≤ V_n}.
3. Compute NLCR = Σ_{n∈n_L} IMF₁²(n) / Σ_{n∈n_L} x²(n).

**D. Adaptive IMF Selection**  
- If NLCR ≤ β (β = 0.02): IMF₁ is noise-dominated → merge: IMF = IMF₁ + IMF₂; Residue R = x − IMF.
- Otherwise: IMF = IMF₁; Residue R = x − IMF₁ − IMF₂.

**E. DFT-based Feature Vectors**  
Apply `scipy.fftpack.fft` to the filtered signal, IMF, and Residue. Then compute element-wise frequency-domain similarity (§ Feature Extraction):  
- IMF_similarity[i] = |Signal_DFT[i]| × |IMF_DFT[i]| / (||Signal_DFT|| × ||IMF_DFT||)  
- R_similarity[i]   = |Signal_DFT[i]| × |R_DFT[i]|   / (||Signal_DFT|| × ||R_DFT||)  

These two vectors (each of length N = 1250) are concatenated into a single 2500-dimensional feature vector.

**F. Save**  
Each feature set is serialised as a `Features` object and saved to `Pickles/MITMVAdbFFT/` or `Pickles/cudbFFT/`.

> Batch runs: 282 MITMVA episodes in ~39 min; 40 CUDB episodes in ~10 min (EMD is computationally expensive — O(N log N) per decomposition).

In [8]:
from signal_processing import batchSignalProcessing

batchSignalProcessing(5)

100%|██████████| 40/40 [09:56<00:00, 14.90s/it]


## Annotating

Annotating all the ECG Episodes

### Overview
After signal processing has extracted frequency-domain features, this step enriches each serialised `Features` object with a **multi-threshold VF label**. The label encodes whether the episode should be classified as VF at 10%, 20%, …, 100% thresholds of VF content — enabling experiments with different strictness levels. In the final experiments, `percent=100` is used, meaning an episode is labelled VF only if 100% of its annotated beats within the window are VF.

### Step-by-step (`data_handling.labelEpisodes` → `labelMITMVADBEpisodes` + `labelCUDBEpisodes`)

1. **Load the feature pickle** from `Pickles/MITMVAdbFFT/F<file>E<episode>C<channel>.p`.
2. **Load the corresponding episode pickle** from `Pickles/MITMVAdb/` to recover the per-sample beat annotations.
3. **Create annotation array** (`createAnnotationArray`): expand the indexed annotation (a sparse list of `(sample_index, label)` pairs) into a dense per-sample array covering all 1250 samples of the episode.
4. **Distribute annotations across the window** (`createDistributedAnnotations`): use an interval-covering algorithm that assigns a label to each sample based on the nearest annotation marker. This handles the common case where annotations are placed at beat peaks rather than uniformly.
5. **Count VF vs. NotVF samples** in the episode and compute the VF fraction.
6. **Build the 10-element label array** (`createLabelsDict`): `label[k] = 1` if the VF fraction exceeds (k+1)×10%, else 0, for k = 0…9. For example, `label[9]` (the `percent=100` slot used in training) is 1 only if ≥100% of the episode is VF.
7. **For CUDB**, `createCUDBAnnotation` applies specialised parsing: `N` = NSR (not VF), `[` = VF start, `]` = VF end; everything between `[` and `]` markers is labelled VF.
8. **Write the updated Features object** back to its pickle file with the newly populated `label` array.
9. **Result:** 300 MITMVA episodes labelled in ~51 s; 40 CUDB episodes in ~31 s.

> The `percent=100` threshold aligns with the standard practice in the literature: an episode is positive only when the clinician annotations confirm VF throughout the window (§ Dataset, §  SVM Parameter Tuning).

In [2]:
from data_handling import labelEpisodes

labelEpisodes(Te=5)

100%|██████████| 40/40 [00:31<00:00,  1.27it/s]


## Preparing Data

Prepare the data for training and evaluation

### Overview
This step aggregates all per-episode feature pickles into a single dataset file `dataSet.p` ready for machine learning. It reads the DFT-similarity vectors from every processed episode in both databases, applies the binary label (`percent=100` → full-window VF required), and separates examples into two lists: `VF_features` and `notVF_features`. No shuffling or splitting yet — that is done at training time.

### Step-by-step (`machine_learning.createData`)

1. **Iterate over MITMVA episodes** (files 400–699, episodes 0–2099, channel 1): for each existing pickle `Pickles/MITMVAdbFFT/F<i>E<j>C1.p`:
   a. Load the `Features` object.
   b. **Compute IMF_similarity vector** (length N): for each DFT bin i,  
      `IMF_sim[i] = |Signal_DFT[i]| × |IMF1_DFT[i]| / (||Signal_DFT|| × ||IMF1_DFT||)`
   c. **Compute R_similarity vector** (length N) analogously with R_DFT.
   d. **Concatenate** to form a 2N-dimensional feature vector.
   e. **Assign label**: `dataa.label[(percent//10) - 1]` → if 1, append to `VF_features`; else to `notVF_features`.
2. **Repeat for CUDB** episodes (files 0–39, episodes 0–549, channel 1) with the same feature construction.
3. **Save** `(VF_features, notVF_features)` tuple as `dataSet.p` via pickle.

> **Dataset imbalance:** The raw dataset is highly skewed — roughly only 9% of episodes are VF (§ Dataset). This imbalance is addressed later with SMOTE.  
> **Parameters used:** `channel=1` (only channel 1 from VFDB used to avoid redundancy), `percent=100` (strictest VF threshold, entire 5-second window must be VF), both databases enabled.

In [3]:
from machine_learning import createData

createData(channel=1, percent=100 , mitmvadb=True,cudb=True, saveFile='dataSet.p')

## SVM tuning

Performing grid search for SVM parameter tuning

### Overview
Before running cross-validation, the SVM hyper-parameters must be selected. An exhaustive **grid search** over the RBF kernel parameters γ (gamma) and C is performed on a held-out validation split (§ SVM Parameter Tuning). The SVM classifier has two parameters:
- **C (regularisation constant):** Controls the soft/hard margin trade-off. Large C = harder margin, fits training data more closely but may overfit.
- **γ (Gaussian RBF hyperparameter):** Defines the width of the Gaussian kernel: K(x, x') = exp(−γ||x−x'||²). Larger γ = narrower kernel = more localised decision boundary.

The best combination for T_e = 5 s was found to be **γ = 45, C = 100**, yielding a G-Mean Accuracy ≈ 91.97% on the validation set.

### Step-by-step (`machine_learning.svmParameterTuning`)

1. **Load and split data** (`loadData(vfCnt=3000, notVfCnt=5000)`): randomly shuffle, take 3000 VF and 5000 Not-VF samples as training (more Not-VF because that class has greater intra-class variation), leave the rest as validation (VF≈2320, Not-VF≈46087).
2. **Grid search loop**: for each γ ∈ {5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60} × C ∈ {100, 10, 1, 0.1}:
   a. Instantiate `sklearn.svm.SVC(kernel='rbf', gamma=γ, C=C)`.
   b. Fit on the training split.
   c. Evaluate on the validation split using `evaluate()` → compute Specificity, Sensitivity, Accuracy.
   d. Print results in the format `"γ , C : ( Specificity, Sensitivity, Accuracy )"`.
3. **Select best parameters** by inspecting G-Mean Accuracy = √(Sensitivity × Specificity) — since the dataset is imbalanced, a metric that rewards both classes equally is essential (§ Evaluation Metrics).

> From Figure 6 of the paper, γ = 45 and C = 100 consistently outperform other combinations for T_e = 5 s and are propagated to all subsequent experiments.

> **⚠ Parameters are not persisted automatically.** `svmParameterTuning` only prints results to stdout — no file or pickle is written. After inspecting the printed table, the best parameters are **manually hardcoded** as arguments in the subsequent cross-validation cells:
> ```python
> kFoldCrossValidation(gamma=45, C=100, ...)
> stratifiedkFoldCrossValidation(gamma=45, C=100, ...)
> ```
> These values also serve as the default arguments of both functions, so omitting them would still use `gamma=45, C=100`. The selection is a one-time manual step informed by reading the grid-search output; it is not re-derived automatically at runtime.

In [ ]:
from machine_learning import svmParameterTuning

svmParameterTuning(file='dataSet.p',vfCnt=3000,notvfCnt=5000)

"5 , 100 : ( 91.73302666695598 , 90.60344827586206 , 91.67888941681989"
"5 , 10 : ( 90.40293358213813 , 92.32758620689656 , 90.49517631747474"
"5 , 1 : ( 89.37010436782607 , 89.00862068965517 , 89.35277955667569"
"5 , 0.1 : ( 89.15095363117582 , 83.27586206896552 , 88.86937839568657"
"10 , 100 : ( 91.9044415995834 , 90.38793103448276 , 91.83175986944038"
"10 , 10 : ( 91.39019680170114 , 92.62931034482759 , 91.44958373788914"


## SMOTE

Creating synthetic data to remove class imbalance

### Overview
The raw VF/Not-VF dataset is severely imbalanced (~9% VF). Training a classifier directly on imbalanced data biases it toward predicting "Not VF" — it can achieve high *accuracy* while having poor *sensitivity* (missing real VF events), which is clinically unacceptable. To overcome this, the authors use **SMOTE (Synthetic Minority Over-Sampling Technique)** developed by Chawla et al. (2002) rather than simple random duplication, which risks overfitting.

SMOTE generates new synthetic VF samples by interpolating between existing minority-class points in feature space, effectively spreading the probability mass over the neighbourhood of real VF samples instead of duplicating them verbatim (§ Overcoming the Imbalance in the Dataset).

### Step-by-step (`machine_learning.upsampleSMOTE`)

1. **Load** `dataSet.p` → `(VF_features, notVF_features)`.
2. **Assemble** a combined array X with labels Y (1 = VF, 0 = Not VF).
3. **Apply SMOTE**: `imblearn.over_sampling.SMOTE(kind='regular').fit_sample(X, Y)` — synthesises new VF feature vectors by:
   - For each VF sample, identify its k nearest VF neighbours in the 2N-dimensional feature space.
   - Interpolate: `new_sample = sample + λ × (neighbour − sample)` where λ ∈ [0, 1] is random.
   - Repeat until VF count matches Not-VF count.
4. **Split** the upsampled array back into `VF_features` and `notVF_features` lists.
5. **Save** to `smoteData.p` for use in the cross-validation steps.

> After SMOTE, both classes have equal cardinality, enabling unbiased classifier training. The cross-validation experiments are performed on this balanced dataset.

In [2]:
from machine_learning import upsampleSMOTE

upsampleSMOTE(loadFile='dataSet.p',saveFile='smoteData.p')

## Feature Ranking

Computing the feature importance

### Overview
The raw feature vector has dimension **2N** (N frequency bins for IMF_similarity concatenated with N bins for R_similarity). For T_e = 5 s at 250 Hz, N = 1250, giving **2500 features**. Not all frequency components are informative: high-frequency bins correspond to noise above the ECG band (filtered to 20 Hz), and DFT coefficients of real signals are symmetric so the left half mirrors the right half. 

A **Random Forest** of 750 decision trees (Breiman, 2001; Ho, 1995) is used to rank all 2500 features by their *importance* (mean decrease in impurity across all trees and splits). The top 24% of features (≈ 600 features) are retained for the SVM — this reduces dimensionality and was shown to improve G-Mean Accuracy slightly (0.002%) compared to using only the top 16% (§ Feature Ranking by Random Forest Classifier, Figure 7).

### Step-by-step

1. **Load data** (`loadData(vfCnt=3000, notVfCnt=5000, file='dataSet.p')`): randomly sample 3000 VF and 5000 Not-VF episodes as training, remainder as validation. Returns `(X_Train, Y_Train, X_Test, Y_Test)`.
2. **Train Random Forest** (`featureRanking(X_Train, Y_Train)`):
   - `RandomForestClassifier(n_estimators=750, random_state=3)` — 750 trees for stable importance estimates.
   - Fit on the training data.
   - Save the trained forest to `randomForest750.p`.
3. **Extract importances**: `forest.feature_importances_` gives a 2500-element importance vector.
4. **Rank features**: `indices = np.argsort(importances)[::-1]` sorts feature indices from most to least important.
5. **Save ranking**: `featureRanking.p` stores the sorted index array for use during `featureSelection` in the cross-validation steps.

> From the paper's Figure 7a, the most important features cluster around the **centre of the DFT vector** (low-frequency components, 1–5 Hz range), consistent with the theoretical analysis. High-frequency bins (edges of the vector) contribute negligible information.

In [3]:
from machine_learning import loadData, featureRanking

(X,Y,_,_) = loadData(vfCnt=3000,notVfCnt=5000,file='dataSet.p')
featureRanking(X,Y)

building tree 1 of 750
building tree 2 of 750
building tree 3 of 750
building tree 4 of 750
building tree 5 of 750
building tree 6 of 750
building tree 7 of 750
building tree 8 of 750
building tree 9 of 750
building tree 10 of 750
building tree 11 of 750
building tree 12 of 750
building tree 13 of 750
building tree 14 of 750
building tree 15 of 750
building tree 16 of 750
building tree 17 of 750
building tree 18 of 750
building tree 19 of 750
building tree 20 of 750
building tree 21 of 750
building tree 22 of 750
building tree 23 of 750
building tree 24 of 750
building tree 25 of 750
building tree 26 of 750
building tree 27 of 750
building tree 28 of 750
building tree 29 of 750
building tree 30 of 750
building tree 31 of 750
building tree 32 of 750
building tree 33 of 750
building tree 34 of 750
building tree 35 of 750
building tree 36 of 750
building tree 37 of 750
building tree 38 of 750
building tree 39 of 750
building tree 40 of 750


[Parallel(n_jobs=1)]: Done  40 tasks      | elapsed:    9.1s


building tree 41 of 750
building tree 42 of 750
building tree 43 of 750
building tree 44 of 750
building tree 45 of 750
building tree 46 of 750
building tree 47 of 750
building tree 48 of 750
building tree 49 of 750
building tree 50 of 750
building tree 51 of 750
building tree 52 of 750
building tree 53 of 750
building tree 54 of 750
building tree 55 of 750
building tree 56 of 750
building tree 57 of 750
building tree 58 of 750
building tree 59 of 750
building tree 60 of 750
building tree 61 of 750
building tree 62 of 750
building tree 63 of 750
building tree 64 of 750
building tree 65 of 750
building tree 66 of 750
building tree 67 of 750
building tree 68 of 750
building tree 69 of 750
building tree 70 of 750
building tree 71 of 750
building tree 72 of 750
building tree 73 of 750
building tree 74 of 750
building tree 75 of 750
building tree 76 of 750
building tree 77 of 750
building tree 78 of 750
building tree 79 of 750
building tree 80 of 750
building tree 81 of 750
building tree 82

[Parallel(n_jobs=1)]: Done 161 tasks      | elapsed:   35.8s


building tree 162 of 750
building tree 163 of 750
building tree 164 of 750
building tree 165 of 750
building tree 166 of 750
building tree 167 of 750
building tree 168 of 750
building tree 169 of 750
building tree 170 of 750
building tree 171 of 750
building tree 172 of 750
building tree 173 of 750
building tree 174 of 750
building tree 175 of 750
building tree 176 of 750
building tree 177 of 750
building tree 178 of 750
building tree 179 of 750
building tree 180 of 750
building tree 181 of 750
building tree 182 of 750
building tree 183 of 750
building tree 184 of 750
building tree 185 of 750
building tree 186 of 750
building tree 187 of 750
building tree 188 of 750
building tree 189 of 750
building tree 190 of 750
building tree 191 of 750
building tree 192 of 750
building tree 193 of 750
building tree 194 of 750
building tree 195 of 750
building tree 196 of 750
building tree 197 of 750
building tree 198 of 750
building tree 199 of 750
building tree 200 of 750
building tree 201 of 750


[Parallel(n_jobs=1)]: Done 364 tasks      | elapsed:  1.4min


building tree 365 of 750
building tree 366 of 750
building tree 367 of 750
building tree 368 of 750
building tree 369 of 750
building tree 370 of 750
building tree 371 of 750
building tree 372 of 750
building tree 373 of 750
building tree 374 of 750
building tree 375 of 750
building tree 376 of 750
building tree 377 of 750
building tree 378 of 750
building tree 379 of 750
building tree 380 of 750
building tree 381 of 750
building tree 382 of 750
building tree 383 of 750
building tree 384 of 750
building tree 385 of 750
building tree 386 of 750
building tree 387 of 750
building tree 388 of 750
building tree 389 of 750
building tree 390 of 750
building tree 391 of 750
building tree 392 of 750
building tree 393 of 750
building tree 394 of 750
building tree 395 of 750
building tree 396 of 750
building tree 397 of 750
building tree 398 of 750
building tree 399 of 750
building tree 400 of 750
building tree 401 of 750
building tree 402 of 750
building tree 403 of 750
building tree 404 of 750


[Parallel(n_jobs=1)]: Done 647 tasks      | elapsed:  2.3min


building tree 648 of 750
building tree 649 of 750
building tree 650 of 750
building tree 651 of 750
building tree 652 of 750
building tree 653 of 750
building tree 654 of 750
building tree 655 of 750
building tree 656 of 750
building tree 657 of 750
building tree 658 of 750
building tree 659 of 750
building tree 660 of 750
building tree 661 of 750
building tree 662 of 750
building tree 663 of 750
building tree 664 of 750
building tree 665 of 750
building tree 666 of 750
building tree 667 of 750
building tree 668 of 750
building tree 669 of 750
building tree 670 of 750
building tree 671 of 750
building tree 672 of 750
building tree 673 of 750
building tree 674 of 750
building tree 675 of 750
building tree 676 of 750
building tree 677 of 750
building tree 678 of 750
building tree 679 of 750
building tree 680 of 750
building tree 681 of 750
building tree 682 of 750
building tree 683 of 750
building tree 684 of 750
building tree 685 of 750
building tree 686 of 750
building tree 687 of 750


[Parallel(n_jobs=1)]: Done 750 out of 750 | elapsed:  2.7min finished


## 10 Fold Cross Validation

Performing the 10 fold cross validation

### Overview
The main evaluation protocol is **10-fold cross-validation** on the SMOTE-balanced dataset, following Kohavi (1995) who showed it to be the best cross-validation scheme for real-world data. The full balanced dataset is randomly shuffled and split into 10 equal folds; in each round, 9 folds train the SVM and 1 fold is held out for testing. This process removes human selection bias and accounts for variance in the data.

**Parameters fixed from previous steps:** γ = 45, C = 100 (from SVM tuning), featurePercent = 24% (from feature ranking), file = `smoteData.p` (SMOTE-balanced).

### Step-by-step (`machine_learning.kFoldCrossValidation`)

1. **Load** `smoteData.p` → combine VF (label=1) and Not-VF (label=0) into arrays X (features), Y (labels).
2. **Feature selection** (`featureSelection(X, percentage=24)`): load `featureRanking.p`, keep only the top 24% × 2500 = **600 highest-ranked features** per sample. This reduces the dimensionality from 2500 → 600.
3. **K-Fold split** (`KFold(n_splits=10, shuffle=True, random_state=3)`): creates 10 randomised, non-stratified folds.
4. **For each fold k = 1…10:**
   a. Train `SVC(kernel='rbf', gamma=45, C=100)` on the 9 training folds.
   b. Evaluate on the training folds → report Specificity, Sensitivity, Accuracy (always ~100% — confirms the model fits training data).
   c. Evaluate on the held-out test fold → compute TP, FP, TN, FN and report Specificity, Sensitivity, Accuracy.
5. **Aggregate across folds.**

### Results (§ K-Fold Cross-Validation, Table 1)
| Metric | Mean | Std |
|---|---|---|
| Sensitivity | 99.988% | ±0.016% |
| Specificity | 98.401% | ±0.19% |
| Accuracy | 99.194% | ±0.092% |
| G-Mean Accuracy | 99.191% | ±0.095% |

   
> **Note on false positives:** Most false positives were found to contain a small VF segment appearing 1–2 seconds *after* the episode window — making them early-warning true positives in a real-time system context.

In [ ]:
from machine_learning import kFoldCrossValidation

kFoldCrossValidation(gamma=45,C=100,file='smoteData.p',featurePercent=24)

**********************
1


## Stratified 10 Fold Cross Validation

Performing the stratified 10 fold cross validation

### Overview
This step repeats 10-fold cross-validation using **stratified** folds: each fold maintains the same class distribution as the full dataset (Kohavi, 1995). Stratification ensures that neither fold is accidentally dominated by one class — important even on the SMOTE-balanced dataset because the shuffled ordering may produce slight imbalances at fold boundaries.

Identical hyper-parameters are used (γ = 45, C = 100, featurePercent = 24%, `smoteData.p`), so results can be directly compared to the non-stratified run above.

### Step-by-step (`machine_learning.stratifiedkFoldCrossValidation`)

1. **Load** `smoteData.p` → X (features), Y (labels), same as before.
2. **Feature selection**: same `featureSelection(X, percentage=24)` → 600-feature vectors.
3. **Stratified K-Fold split** (`StratifiedKFold(n_splits=10, shuffle=True, random_state=3)`): ensures each fold contains approximately the same proportion of VF and Not-VF samples.
4. **For each fold k = 1…10:**
   a. Train `SVC(kernel='rbf', gamma=45, C=100)` on the 9 training folds.
   b. Evaluate training performance (consistently ~100%).
   c. Evaluate held-out test fold → Specificity, Sensitivity, Accuracy.

### Results (§ K-Fold Cross-Validation, Table 1)
| Metric | Mean | Std |
|---|---|---|
| Sensitivity | 99.992% | ±0.010% |
| Specificity | 98.395% | ±0.187% |
| Accuracy | 99.194% | ±0.092% |
| G-Mean Accuracy | 99.190% | ±0.096% |

Results are virtually identical to the non-stratified run, confirming the robustness of VFPred across different data partitioning strategies. The slight increase in Sensitivity (99.992% vs 99.988%) and marginal decrease in Specificity (98.395% vs 98.401%) reflect the effect of more balanced test folds.

> **Comparison with the state-of-the-art:** VFPred's G-Mean Accuracy of 99.19% substantially outperforms all prior methods: Asl et al. (2008) achieved 97.57% (but requiring ~30 s windows); Verma et al. (2016) achieved 94.91% for 5 s; Qiao et al. (2014) achieved 96.20% (Table 2 of the paper).

In [ ]:
from machine_learning import stratifiedkFoldCrossValidation

stratifiedkFoldCrossValidation(gamma=45,C=100,file='smoteData.p',featurePercent=24)